# BigQuery Anti-Pattern Recognition - Cloud Run API Demo

This notebook demonstrates how to use the BigQuery Anti-Pattern Recognition tool via the Cloud Run REST API.

## What this notebook does:
1. Load configuration from the setup notebook
2. Test the Cloud Run API with sample queries
3. Demonstrate anti-pattern detection
4. Show AI-powered query rewriting
5. Analyze performance improvements
6. Batch process multiple queries

## Prerequisites:
- Complete `01_setup_and_deploy.ipynb` first
- Cloud Run service must be deployed and running

---

## Step 1: Load Configuration and Setup

In [ ]:
# Import required libraries
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Import our utilities
from utils import (
    ConfigManager, 
    AntiPatternAnalyzer, 
    ResultsFormatter, 
    SampleQueries,
    format_sql,
    create_diff_view
)

print("✅ Libraries imported successfully!")

In [ ]:
# Load configuration from setup notebook
config = ConfigManager()

# Verify deployment is complete
if config.get('deployment_status') != 'completed':
    print("❌ Setup not complete. Please run 01_setup_and_deploy.ipynb first.")
else:
    print("✅ Configuration loaded successfully!")
    print(f"Service URL: {config.get('service_url')}")
    print(f"Project: {config.get('project_id')}")
    print(f"Region: {config.get('region')}")

# Initialize analyzer
analyzer = AntiPatternAnalyzer(config)
print("✅ Anti-pattern analyzer initialized!")

## Step 2: Interactive Query Testing

Let's create an interactive interface to test different queries:

In [ ]:
# Create interactive query tester
sample_queries = SampleQueries.get_all_queries()

# Widgets for query selection and input
query_selector = widgets.Dropdown(
    options=[('Custom Query', 'custom')] + [(v['name'], k) for k, v in sample_queries.items()],
    value='select_star',
    description='Select Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

query_input = widgets.Textarea(
    value=sample_queries['select_star']['query'],
    description='SQL Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='150px')
)

rewrite_checkbox = widgets.Checkbox(
    value=False,
    description='Enable AI Query Rewriting',
    style={'description_width': 'initial'}
)

analyze_button = widgets.Button(
    description='🔍 Analyze Query',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

output_area = widgets.Output()

def on_query_change(change):
    if change['new'] != 'custom':
        query_input.value = sample_queries[change['new']]['query']

def on_analyze_click(b):
    with output_area:
        clear_output()
        print("🔄 Analyzing query...")
        
        try:
            # Call API
            result = analyzer.call_api(
                query_input.value, 
                rewrite_sql=rewrite_checkbox.value
            )
            
            if 'error' in result:
                print(f"❌ Error: {result['error']}")
                return
            
            # Parse and display results
            antipatterns = ResultsFormatter.parse_antipatterns(result)
            
            print("\n📊 Analysis Results:")
            print("=" * 50)
            
            if antipatterns:
                for i, ap in enumerate(antipatterns, 1):
                    print(f"\n{i}. {ap['name']}")
                    print(f"   {ap['description']}")
            else:
                print("✅ No anti-patterns detected!")
            
            # Show rewritten query if available
            if rewrite_checkbox.value and 'optimized_sql' in result:
                print("\n🤖 AI-Optimized Query:")
                print("=" * 50)
                print(result['optimized_sql'])
                
                # Create diff view
                diff_html = create_diff_view(query_input.value, result['optimized_sql'])
                display(HTML(diff_html))
            
            # Cost estimation
            print("\n💰 Cost Analysis:")
            print("=" * 50)
            
            original_cost = analyzer.estimate_query_cost(query_input.value)
            if 'error' not in original_cost:
                print(f"Original query: {original_cost['gb_processed']:.2f} GB processed")
                print(f"Estimated cost: ${original_cost['estimated_cost_usd']:.4f}")
                
                if rewrite_checkbox.value and 'optimized_sql' in result:
                    optimized_cost = analyzer.estimate_query_cost(result['optimized_sql'])
                    if 'error' not in optimized_cost:
                        print(f"Optimized query: {optimized_cost['gb_processed']:.2f} GB processed")
                        print(f"Estimated cost: ${optimized_cost['estimated_cost_usd']:.4f}")
                        
                        savings = original_cost['gb_processed'] - optimized_cost['gb_processed']
                        if savings > 0:
                            savings_pct = (savings / original_cost['gb_processed']) * 100
                            print(f"\n💡 Potential savings: {savings:.2f} GB ({savings_pct:.1f}%)")
            
        except Exception as e:
            print(f"❌ Error analyzing query: {e}")

query_selector.observe(on_query_change, names='value')
analyze_button.on_click(on_analyze_click)

# Display interface
display(widgets.VBox([
    widgets.HTML("<h3>🔍 Interactive Query Analyzer</h3>"),
    query_selector,
    query_input,
    rewrite_checkbox,
    analyze_button,
    output_area
]))

## Step 3: Batch Analysis Demo

Let's analyze all sample queries at once and create visualizations:

In [ ]:
# Batch analyze all sample queries
print("🔄 Analyzing all sample queries...")

results = []
sample_queries = SampleQueries.get_all_queries()

for query_key, query_info in sample_queries.items():
    print(f"Analyzing: {query_info['name']}")
    
    try:
        # Analyze without rewriting first
        result = analyzer.call_api(query_info['query'], rewrite_sql=False)
        
        if 'error' not in result:
            antipatterns = ResultsFormatter.parse_antipatterns(result)
            
            # Get cost estimation
            cost_info = analyzer.estimate_query_cost(query_info['query'])
            
            results.append({
                'query_key': query_key,
                'query_name': query_info['name'],
                'description': query_info['description'],
                'antipatterns_found': len(antipatterns),
                'antipattern_names': [ap['name'] for ap in antipatterns],
                'gb_processed': cost_info.get('gb_processed', 0) if 'error' not in cost_info else 0,
                'estimated_cost': cost_info.get('estimated_cost_usd', 0) if 'error' not in cost_info else 0
            })
        else:
            print(f"  ❌ Error: {result['error']}")
            
    except Exception as e:
        print(f"  ❌ Exception: {e}")

# Create DataFrame for analysis
df_results = pd.DataFrame(results)
print(f"\n✅ Analyzed {len(results)} queries successfully!")

# Display results table
display(HTML("<h3>📊 Batch Analysis Results</h3>"))
display(df_results[['query_name', 'antipatterns_found', 'gb_processed', 'estimated_cost']])

## Step 4: Visualizations

Create charts to visualize the analysis results:

In [ ]:
# Create visualizations
if not df_results.empty:
    # 1. Anti-patterns by query
    fig1 = px.bar(
        df_results, 
        x='query_name', 
        y='antipatterns_found',
        title='Anti-Patterns Found by Query',
        labels={'antipatterns_found': 'Number of Anti-Patterns', 'query_name': 'Query'},
        color='antipatterns_found',
        color_continuous_scale='Reds'
    )
    fig1.update_xaxis(tickangle=45)
    fig1.show()
    
    # 2. Cost distribution
    fig2 = px.scatter(
        df_results,
        x='gb_processed',
        y='estimated_cost',
        size='antipatterns_found',
        hover_name='query_name',
        title='Query Cost vs Data Processed',
        labels={'gb_processed': 'GB Processed', 'estimated_cost': 'Estimated Cost (USD)'}
    )
    fig2.show()
    
    # 3. Anti-pattern frequency
    all_antipatterns = []
    for ap_list in df_results['antipattern_names']:
        all_antipatterns.extend(ap_list)
    
    if all_antipatterns:
        ap_counts = pd.Series(all_antipatterns).value_counts()
        
        fig3 = px.pie(
            values=ap_counts.values,
            names=ap_counts.index,
            title='Anti-Pattern Distribution'
        )
        fig3.show()
    
    print("📈 Visualizations created successfully!")
else:
    print("⚠️ No results to visualize")

## Step 5: AI Rewriting Comparison

Let's test the AI rewriting feature on a problematic query:

In [ ]:
# Test AI rewriting on a complex query
test_query = sample_queries['multiple_ctes']['query']

print("🤖 Testing AI Query Rewriting")
print("=" * 50)
print("\n📝 Original Query:")
print(format_sql(test_query))

# Analyze with rewriting enabled
print("\n🔄 Calling API with rewriting enabled...")
try:
    result = analyzer.call_api(test_query, rewrite_sql=True)
    
    if 'error' in result:
        print(f"❌ Error: {result['error']}")
    else:
        # Show anti-patterns
        antipatterns = ResultsFormatter.parse_antipatterns(result)
        print(f"\n🔍 Anti-patterns found: {len(antipatterns)}")
        for ap in antipatterns:
            print(f"  • {ap['name']}: {ap['description']}")
        
        # Show optimized query
        if 'optimized_sql' in result and result['optimized_sql']:
            print("\n✨ AI-Optimized Query:")
            print(format_sql(result['optimized_sql']))
            
            # Cost comparison
            print("\n💰 Cost Comparison:")
            original_cost = analyzer.estimate_query_cost(test_query)
            optimized_cost = analyzer.estimate_query_cost(result['optimized_sql'])
            
            if 'error' not in original_cost and 'error' not in optimized_cost:
                print(f"Original: {original_cost['gb_processed']:.2f} GB (${original_cost['estimated_cost_usd']:.4f})")
                print(f"Optimized: {optimized_cost['gb_processed']:.2f} GB (${optimized_cost['estimated_cost_usd']:.4f})")
                
                if original_cost['gb_processed'] > optimized_cost['gb_processed']:
                    savings = original_cost['gb_processed'] - optimized_cost['gb_processed']
                    savings_pct = (savings / original_cost['gb_processed']) * 100
                    print(f"💡 Savings: {savings:.2f} GB ({savings_pct:.1f}%)")
                    
                    # Create comparison chart
                    fig = go.Figure(data=[
                        go.Bar(name='Original', x=['GB Processed'], y=[original_cost['gb_processed']], marker_color='red'),
                        go.Bar(name='Optimized', x=['GB Processed'], y=[optimized_cost['gb_processed']], marker_color='green')
                    ])
                    fig.update_layout(title='Query Optimization Results', barmode='group')
                    fig.show()
        else:
            print("\n⚠️ No optimized query returned")
            
except Exception as e:
    print(f"❌ Error during rewriting: {e}")

## Step 6: Performance Metrics

Let's measure API performance and response times:

In [ ]:
import time

# Performance testing
print("⏱️ Performance Testing")
print("=" * 50)

performance_results = []
test_queries = list(sample_queries.items())[:3]  # Test first 3 queries

for query_key, query_info in test_queries:
    print(f"\nTesting: {query_info['name']}")
    
    # Test without rewriting
    start_time = time.time()
    result = analyzer.call_api(query_info['query'], rewrite_sql=False)
    end_time = time.time()
    
    if 'error' not in result:
        response_time = end_time - start_time
        antipatterns = ResultsFormatter.parse_antipatterns(result)
        
        performance_results.append({
            'query_name': query_info['name'],
            'response_time': response_time,
            'antipatterns_found': len(antipatterns),
            'rewriting_enabled': False
        })
        
        print(f"  ✅ Response time: {response_time:.2f}s, Anti-patterns: {len(antipatterns)}")
    else:
        print(f"  ❌ Error: {result['error']}")
    
    # Test with rewriting (slower)
    start_time = time.time()
    result = analyzer.call_api(query_info['query'], rewrite_sql=True)
    end_time = time.time()
    
    if 'error' not in result:
        response_time = end_time - start_time
        antipatterns = ResultsFormatter.parse_antipatterns(result)
        
        performance_results.append({
            'query_name': query_info['name'],
            'response_time': response_time,
            'antipatterns_found': len(antipatterns),
            'rewriting_enabled': True
        })
        
        print(f"  ✅ Response time (with rewriting): {response_time:.2f}s")
    else:
        print(f"  ❌ Error with rewriting: {result['error']}")

# Visualize performance results
if performance_results:
    df_perf = pd.DataFrame(performance_results)
    
    fig = px.bar(
        df_perf,
        x='query_name',
        y='response_time',
        color='rewriting_enabled',
        title='API Response Times',
        labels={'response_time': 'Response Time (seconds)', 'query_name': 'Query'},
        barmode='group'
    )
    fig.update_xaxis(tickangle=45)
    fig.show()
    
    print("\n📊 Performance Summary:")
    avg_time_no_rewrite = df_perf[df_perf['rewriting_enabled'] == False]['response_time'].mean()
    avg_time_with_rewrite = df_perf[df_perf['rewriting_enabled'] == True]['response_time'].mean()
    
    print(f"Average response time (detection only): {avg_time_no_rewrite:.2f}s")
    print(f"Average response time (with rewriting): {avg_time_with_rewrite:.2f}s")
    print(f"Rewriting overhead: {avg_time_with_rewrite - avg_time_no_rewrite:.2f}s")

## Step 7: Export Results

Save analysis results for further use:

In [ ]:
# Export results to CSV
if not df_results.empty:
    output_file = 'api_demo_results.csv'
    df_results.to_csv(output_file, index=False)
    print(f"✅ Results exported to {output_file}")
    
    # Create summary report
    summary_html = f"""
    <div style="background-color: #f0f8ff; padding: 20px; border-radius: 10px; margin: 10px 0;">
        <h3>📋 API Demo Summary</h3>
        
        <h4>🔍 Analysis Results:</h4>
        <ul>
            <li><strong>Queries Analyzed:</strong> {len(df_results)}</li>
            <li><strong>Total Anti-patterns Found:</strong> {df_results['antipatterns_found'].sum()}</li>
            <li><strong>Average Anti-patterns per Query:</strong> {df_results['antipatterns_found'].mean():.1f}</li>
            <li><strong>Total Data Processed:</strong> {df_results['gb_processed'].sum():.2f} GB</li>
            <li><strong>Total Estimated Cost:</strong> ${df_results['estimated_cost'].sum():.4f}</li>
        </ul>
        
        <h4>🏆 Most Problematic Queries:</h4>
        <ol>
    """
    
    # Add top 3 most problematic queries
    top_queries = df_results.nlargest(3, 'antipatterns_found')
    for _, row in top_queries.iterrows():
        summary_html += f"<li>{row['query_name']} ({row['antipatterns_found']} anti-patterns)</li>"
    
    summary_html += """
        </ol>
        
        <h4>🚀 Next Steps:</h4>
        <ul>
            <li>Run <strong>03_bigquery_udf_demo.ipynb</strong> to test the UDF interface</li>
            <li>Run <strong>04_streamlit_frontend.ipynb</strong> to deploy the web interface</li>
            <li>Use the API in your own applications</li>
        </ul>
    </div>
    """
    
    display(HTML(summary_html))
else:
    print("⚠️ No results to export")

print("\n🎯 Cloud Run API demo completed successfully!")
print("\nYou can now:")
print("1. Integrate the API into your applications")
print("2. Set up automated query analysis pipelines")
print("3. Use the API for real-time query optimization")
print("4. Proceed to the next demo notebook")